#데이터 분석 미니 프로젝트

#실 거주지 인구 데이터 수집 및 현황 분석하기(데이터 인풋받기)

In [52]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb

In [53]:
# 한글 폰트 설정
import platform

from matplotlib import rc
plt.rcParams['axes.unicode_minus'] = False

if platform.system() == 'Linux':
    rc('font', family = 'NanumGothic')  # 또는 '나눔고딕'
    print('Linux system... font set to NanumGothic')                    
elif platform.system() == 'Windows':
    rc('font', family = 'Malgun Gothic')   # 또는 '맑은 고딕'
    print('Windows system... font set to Malgun Gothic')
else:
    print('Unknown system... sorry~~~~')

Linux system... font set to NanumGothic


In [54]:
pb = pd.read_csv('datas/2425_vc.csv',encoding='cp949') 
pb['행정구역'] = pb['행정구역'].str.replace(r'[^가-힣]', '', regex=True)

pb.head()


,행정구역,2024년_거주자_총인구수,2024년_거주자_연령구간인구수,2024년_거주자_0~9세,2024년_거주자_10~19세,2024년_거주자_20~29세,2024년_거주자_30~39세,2024년_거주자_40~49세,2024년_거주자_50~59세,2024년_거주자_60~69세,...,2025년_여_거주자_10~19세,2025년_여_거주자_20~29세,2025년_여_거주자_30~39세,2025년_여_거주자_40~49세,2025년_여_거주자_50~59세,2025년_여_거주자_60~69세,2025년_여_거주자_70~79세,2025년_여_거주자_80~89세,2025년_여_거주자_90~99세,2025년_여_거주자_100세 이상
0,경기도성남시,"907,236","907,236","56,953","81,501","111,566","133,888","142,375","151,623","128,104",...,"39,730","51,659","64,714","70,456","75,150","66,313","38,186","18,917","4,058",110
1,경기도성남시수정구,"233,461","233,461","13,468","16,141","29,679","35,958","34,250","38,500","36,751",...,"7,891","13,297","16,409","15,838","18,375","18,828","11,448","5,145",925,20
2,경기도성남시수정구신흥동,"12,210","12,210",285,585,"1,549","1,639","1,624","2,303","2,429",...,279,656,634,635,940,"1,185",740,338,56,1
3,경기도성남시수정구신흥동,"31,264","31,264","3,016","2,179","3,316","6,058","4,892","4,733","4,259",...,"1,111","1,533","3,019","2,460","2,554","2,347","1,186",420,81,4
4,경기도성남시수정구신흥동,"10,612","10,612",282,508,"1,398","1,518","1,421","1,993","2,000",...,238,622,633,578,859,946,569,285,39,1


In [55]:
# 년도별 합계 추가하기

years = set(col.split()[0] for col in df.columns)

for year in years:
    cols = [col for col in df.columns if col.startswith(year)]
    df[f"{year} 합계"] = df[cols].sum(axis=1)

#미래예측
전년도와 해당년도를 비교하여 다음년도 인구 분포를 체크
전년도 = a
해당년도 = b

rate = (b - a) / a #증감률
c = b * (1 + rate) #내년도 예산값

In [56]:
# pb['외국인비율'] =pop_Seoul['외국인']/pop_Seoul['인구수']*100

# Primary, secondary, third 로 나이마다 나눔
# Primary = 연령 76세 이상군
# Secondary = 61~75세
# Tertiary = 11~60
# other = 9세 이하

In [57]:
pb_result = pb.copy()

groups = {
    "Primary": ["70~79세", "80~89세", "90~99세", "100세 이상"],
    "Secondary": ["60~69세"],
    "Tertiary": [
        "10~19세",
        "20~29세",
        "30~39세",
        "40~49세",
        "50~59세"
    ],
    "Other": ["0~9세"]
}

# 연도 자동 추출
years = sorted({
    col.split("년")[0]
    for col in pb.columns
    if "년_거주자_" in col
})

for year in years:

    # 해당 연도의 총인구 컬럼
    total_col = f"{year}년_거주자_총인구수"

    for group, ages in groups.items():

        # 그룹에 해당하는 컬럼 찾기
        cols = [
            f"{year}년_거주자_{age}"
            for age in ages
            if f"{year}년_거주자_{age}" in pb.columns
        ]

        if cols:

            # 그룹 인구수
            group_data = (
                pb[cols]
                .replace(",", "", regex=True)
                .apply(pd.to_numeric, errors="coerce")
                .sum(axis=1)
            )

            # 그룹 인구수 추가
            pb_result[f"{year}년_{group}"] = group_data

            # 총인구수 숫자 변환
            total_data = pd.to_numeric(
                pb[total_col].astype(str).str.replace(",", ""),
                errors="coerce"
            )

            # 총인구 대비 비율(%)
            pb_result[f"{year}년_{group}_비율"] = (
                group_data / total_data * 100
            ).round(2)

pb_result.head()

,행정구역,2024년_거주자_총인구수,2024년_거주자_연령구간인구수,2024년_거주자_0~9세,2024년_거주자_10~19세,2024년_거주자_20~29세,2024년_거주자_30~39세,2024년_거주자_40~49세,2024년_거주자_50~59세,2024년_거주자_60~69세,...,2024년_Other,2024년_Other_비율,2025년_Primary,2025년_Primary_비율,2025년_Secondary,2025년_Secondary_비율,2025년_Tertiary,2025년_Tertiary_비율,2025년_Other,2025년_Other_비율
0,경기도성남시,"907,236","907,236","56,953","81,501","111,566","133,888","142,375","151,623","128,104",...,56953,6.28,107920,11.99,129153,14.35,608957,67.65,54127,6.01
1,경기도성남시수정구,"233,461","233,461","13,468","16,141","29,679","35,958","34,250","38,500","36,751",...,13468,5.77,30808,13.33,36998,16.01,150567,65.16,12688,5.49
2,경기도성남시수정구신흥동,"12,210","12,210",285,585,"1,549","1,639","1,624","2,303","2,429",...,285,2.33,1918,16.03,2389,19.97,7418,62.00,239,2.00
3,경기도성남시수정구신흥동,"31,264","31,264","3,016","2,179","3,316","6,058","4,892","4,733","4,259",...,3016,9.65,3120,9.90,4360,13.84,20989,66.63,3031,9.62
4,경기도성남시수정구신흥동,"10,612","10,612",282,508,"1,398","1,518","1,421","1,993","2,000",...,282,2.66,1599,15.15,2026,19.19,6666,63.15,265,2.51


# 증감률 = (현재년도 - 전년도) / 전년도

# 다음년도 예상값 = 현재년도 × (1 + 증감률)

In [58]:
# ==========================================
# 다음 연도 인구수 예측
# 알고리즘 : 전년 대비 증감률
# ==========================================

groups = ["Primary", "Secondary", "Tertiary", "Other"]

# 연도 자동 추출
years = sorted({
    col.split("년")[0]
    for col in pb_result.columns
    if "년_" in col
})

# 가장 최근 2개 연도
previous_year = years[-2]
current_year = years[-1]

# 예측할 다음 연도
next_year = str(int(current_year) + 1)

print(f"기준 : {previous_year} → {current_year}")
print(f"예측 : {next_year}")


# ==========================================
# 1. 그룹별 다음 연도 예상 인구
# ==========================================

for group in groups:

    previous_col = f"{previous_year}년_{group}"
    current_col = f"{current_year}년_{group}"
    predict_col = f"{next_year}년_{group}_예상"

    # 전년 대비 증감률
    growth_rate = (
        (pb_result[current_col] - pb_result[previous_col])
        / pb_result[previous_col]
    )

    # 다음 연도 예상 인구
    pb_result[predict_col] = (
        pb_result[current_col] * (1 + growth_rate)
    ).round(0)


# ==========================================
# 2. 다음 연도 예상 총인구
# ==========================================

predict_cols = [
    f"{next_year}년_{group}_예상"
    for group in groups
]

pb_result[f"{next_year}년_총인구수_예상"] = (
    pb_result[predict_cols].sum(axis=1)
)


# ==========================================
# 3. 그룹별 예상 인구 비율
# ==========================================

predict_total = pb_result[f"{next_year}년_총인구수_예상"]

for group in groups:

    predict_col = f"{next_year}년_{group}_예상"
    ratio_col = f"{next_year}년_{group}_예상비율"

    pb_result[ratio_col] = (
        pb_result[predict_col]
        / predict_total
        * 100
    ).round(2)


# 결과 확인
pb_result.head()

기준 : 2024 → 2025
예측 : 2026


,행정구역,2024년_거주자_총인구수,2024년_거주자_연령구간인구수,2024년_거주자_0~9세,2024년_거주자_10~19세,2024년_거주자_20~29세,2024년_거주자_30~39세,2024년_거주자_40~49세,2024년_거주자_50~59세,2024년_거주자_60~69세,...,2025년_Other_비율,2026년_Primary_예상,2026년_Secondary_예상,2026년_Tertiary_예상,2026년_Other_예상,2026년_총인구수_예상,2026년_Primary_예상비율,2026년_Secondary_예상비율,2026년_Tertiary_예상비율,2026년_Other_예상비율
0,경기도성남시,"907,236","907,236","56,953","81,501","111,566","133,888","142,375","151,623","128,104",...,6.01,115057.0,130211.0,597193.0,51441.0,893902.0,12.87,14.57,66.81,5.75
1,경기도성남시수정구,"233,461","233,461","13,468","16,141","29,679","35,958","34,250","38,500","36,751",...,5.49,33055.0,37247.0,146708.0,11953.0,228963.0,14.44,16.27,64.07,5.22
2,경기도성남시수정구신흥동,"12,210","12,210",285,585,"1,549","1,639","1,624","2,303","2,429",...,2.00,2048.0,2350.0,7146.0,200.0,11744.0,17.44,20.01,60.85,1.70
3,경기도성남시수정구신흥동,"31,264","31,264","3,016","2,179","3,316","6,058","4,892","4,733","4,259",...,9.62,3463.0,4463.0,20802.0,3046.0,31774.0,10.90,14.05,65.47,9.59
4,경기도성남시수정구신흥동,"10,612","10,612",282,508,"1,398","1,518","1,421","1,993","2,000",...,2.51,1714.0,2052.0,6498.0,249.0,10513.0,16.30,19.52,61.81,2.37


In [59]:
from geopy.geocoders import Nominatim
import pandas as pd
import time

# ==========================================
# 1. 지오코더 생성
# ==========================================

geolocator = Nominatim(
    user_agent="happymaker1024"
)


# ==========================================
# 2. 주소 검색 함수
# ==========================================

def get_location(address):

    # 붙어 있는 행정구역명을 검색하기 쉽게 변환
    search_address = address

    # 주요 행정구역 단위 뒤에 띄어쓰기 추가
    for word in [
        "특별시",
        "광역시",
        "특별자치시",
        "특별자치도",
        "도",
        "시",
        "군",
        "구",
        "읍",
        "면",
        "동"
    ]:
        search_address = search_address.replace(
            word,
            word + " "
        )

    search_address = " ".join(search_address.split())

    try:

        # 대한민국으로 검색 범위 제한
        location = geolocator.geocode(
            search_address,
            country_codes="kr"
        )

        if location:
            return location.latitude, location.longitude

        else:
            return None, None

    except Exception as e:

        print(f"검색 오류: {address} / {e}")

        return None, None


# ==========================================
# 3. df의 index를 이용해서 위도/경도 검색
# ==========================================

# 주소를 위도/경도로 변환
latitudes = []
longitudes = []

for address in pb_result["행정구역"]:

    latitude, longitude = get_location(str(address))

    latitudes.append(latitude)
    longitudes.append(longitude)

    time.sleep(1)


# 위도 / 경도 추가
pb_result["위도"] = latitudes
pb_result["경도"] = longitudes


# CSV 저장
pb_result.to_csv(
    "datas/pb_result.csv",
    index=False,
    encoding="utf-8-sig"
)

# 지도에 표시

In [60]:
# import json
# import folium

# # GeoJSON 불러오기
# with open(
#     "datas/korea_dong.geojson",
#     encoding="utf-8"
# ) as f:
#     geo_data = json.load(f)


# # 지도 생성
# pb_map = folium.Map(
#     location=[37.4, 127.1],
#     zoom_start=10
# )


# # Primary 인구를 영역 색으로 표시
# folium.Choropleth(
#     geo_data=geo_data,

#     # 우리가 만든 데이터
#     data=pb_result,

#     # 행정구역과 Primary 값
#     columns=[
#         "행정구역",
#         "2025년_Primary"
#     ],

#     # GeoJSON 안의 행정구역 이름과 연결
#     key_on="feature.properties.adm_nm",

#     # 색상
#     fill_color="YlOrRd",

#     # 영역 투명도
#     fill_opacity=0.7,

#     # 경계선 투명도
#     line_opacity=0.3,

#     # 범례
#     legend_name="2025년 Primary 인구수"
# ).add_to(pb_map)


# pb_map

In [61]:
import json
import folium
from IPython.display import IFrame

# GeoJSON
with open(
    "datas/HangJeongDong_ver20260701.geojson",
    encoding="utf-8"
) as f:
    geo_data = json.load(f)


# 행정구역 이름 정리
pb_result["지도용_행정구역"] = (
    pb_result["행정구역"]
    .astype(str)
    .str.replace(" ", "", regex=False)
)

for feature in geo_data["features"]:
    feature["properties"]["지도용_행정구역"] = (
        feature["properties"]["adm_nm"]
        .replace(" ", "")
    )


# 지도 생성
pb_map = folium.Map(
    location=[37.4, 127.1],
    zoom_start=11,
    tiles="CartoDB positron"
)


# Primary 영역 표시
folium.Choropleth(
    geo_data=geo_data,
    data=pb_result,

    columns=[
        "지도용_행정구역",
        "2025년_Primary"
    ],

    key_on="feature.properties.지도용_행정구역",

    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.3,

    legend_name="2025년 Primary 인구수"

).add_to(pb_map)


# HTML 저장
pb_map.save(
    "chart_datas/pb_primary_map.html"
)


# Notebook 안에서 지도 표시
IFrame(
    src="chart_datas/pb_primary_map.html",
    width="100%",
    height=600
)